# 02 · Exploratory Data Analysis

Every chart here answers a business question. Charts that would only decorate
the notebook were not made.

Design rules: no dual-axis charts (two y-scales let axis choice manufacture a
relationship), a colour-vision-validated palette, and direct data labels wherever
a colour falls below 3:1 contrast.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from src.config import load_config, get_paths
cfg, paths = load_config(), get_paths()
print("as-of date:", cfg["project"]["as_of_date"])

as-of date: 2011-12-09


In [2]:
from src.database import connect, register_base_tables
con = connect(); register_base_tables(con)
demand_monthly = con.execute("SELECT * FROM demand_monthly").fetchdf()
category_perf  = con.execute("SELECT * FROM category_performance").fetchdf()
abc            = con.execute("SELECT * FROM sku_abc").fetchdf()
drivers        = con.execute("SELECT * FROM stockout_drivers").fetchdf()
stockout_cat   = con.execute("SELECT * FROM stockout_by_category").fetchdf()
con.close()

## Q1 · When does demand happen?

In [3]:
from src.eda import apply_style, fig_seasonality
apply_style()
demand_monthly.assign(month=pd.to_datetime(demand_monthly.month)).set_index("month")[["units","revenue"]].tail(14)

,units,revenue
month,,
2010-11-01,676871.0,1333544.83
2010-12-01,339294.0,727981.35
2011-01-01,300925.0,632981.65
2011-02-01,268631.0,483012.30
2011-03-01,363913.0,657079.77
2011-04-01,281563.0,488655.30
2011-05-01,383872.0,714354.82
2011-06-01,371480.0,662138.38
2011-07-01,362881.0,640073.27


November is the peak at ~2.4x the February trough, and demand collapses to
**34.9%** of peak in the month after Christmas. That collapse is measured, not
assumed, and it drives the forecasting problem in notebook 03: the as-of date
(2011-12-09) sits *on* the peak, so the operational window is the collapse.

## Q2 · Is the demand data dense enough for conventional models?

In [4]:
panel = pd.read_parquet(paths.processed / "demand_daily.parquet")
zero_share = panel.groupby("product_id").demand_units.apply(lambda s: (s==0).mean())
print(zero_share.describe([.1,.25,.5,.75,.9]).round(3))
print(f"\nSKUs zero on >70% of days: {(zero_share>0.7).sum():,} of {len(zero_share):,}")

count    2370.000
mean        0.749
std         0.172
min         0.187
10%         0.478
25%         0.656
50%         0.797
75%         0.881
90%         0.928
max         0.969
Name: demand_units, dtype: float64

SKUs zero on >70% of days: 1,651 of 2,370


**No.** Three-quarters of SKUs are intermittent. This is the evidence that rules out Prophet and routes most SKUs to Croston/SBA.

## Q3 · How concentrated is revenue?

In [5]:
abc.groupby("abc_class").agg(n_skus=("product_id","size"),
                             revenue=("total_revenue","sum")).assign(
    pct_revenue=lambda d: (100*d.revenue/d.revenue.sum()).round(1),
    pct_skus=lambda d: (100*d.n_skus/d.n_skus.sum()).round(1))

,n_skus,revenue,pct_revenue,pct_skus
abc_class,,,,
A,701,13497140.89,80.0,29.6
B,754,2534232.78,15.0,31.8
C,915,844099.79,5.0,38.6


## Q4 · Which categories carry the most demand risk?

In [6]:
category_perf[['category','n_skus','total_revenue','pct_of_revenue','avg_coef_variation','avg_lead_time_days']]

,category,n_skus,total_revenue,pct_of_revenue,avg_coef_variation,avg_lead_time_days
0,Kitchen & Dining,504,4270971.12,25.31,3.298,16.8
1,Home Decor,363,2762387.42,16.37,3.452,16.8
2,Bags,113,1879763.05,11.14,2.753,6.6
3,Party & Celebration,176,1227724.14,7.28,3.186,19.1
4,Storage & Organisation,136,1096606.00,6.50,3.134,9.1
5,Candles & Fragrance,187,1088887.52,6.45,3.732,39.6
6,Other,235,974249.97,5.77,3.336,39.7
7,Christmas & Seasonal,145,824659.08,4.89,3.034,31.6
8,Stationery & Craft,172,671900.26,3.98,3.589,19.1
9,Toys & Games,96,626831.56,3.71,3.049,29.4


## Q5 · What actually drives poor service — lead time or volatility?\n\nThis is the empirical justification for the safety-stock formula used later.

In [7]:
drivers.pivot(index='lead_time_band', columns='volatility_quartile', values='avg_fill_rate').round(3)

volatility_quartile,1,2,3,4
lead_time_band,,,,
1: <=10 days,0.949,0.893,0.897,0.871
2: 11-20 days,0.917,0.919,0.861,0.851
3: 21-30 days,0.886,0.872,0.858,0.836
4: >30 days,0.820,0.857,0.833,0.805


Fill rate degrades along **both** axes, but the volatility gradient is steeper
(94.8% → 75.3%) than the lead-time gradient (89.6% → 83.1%).

That is expected and important: the incumbent policy already covers the
supplier's quoted lead time, so lead time only bites through its *variability*.
Demand variability is completely unbuffered. **The biggest service gains come
from sizing safety stock to demand volatility, not from chasing shorter lead
times** — a conclusion that changes where the business spends effort.

## Q6 · Where is service being lost?

In [8]:
stockout_cat.head(10)

,category,n_skus,n_skus_with_stockouts,demand_units,lost_units,fill_rate,lost_revenue,pct_of_lost_revenue
0,Kitchen & Dining,504,402.0,1531539.0,155909.0,0.8982,389806.66,24.21
1,Home Decor,363,277.0,1135026.0,71209.0,0.9373,188803.23,11.73
2,Christmas & Seasonal,145,122.0,510213.0,108668.0,0.7870,181835.31,11.30
3,Other,235,172.0,815533.0,104695.0,0.8716,150500.59,9.35
4,Stationery & Craft,172,125.0,645973.0,108176.0,0.8325,125879.37,7.82
5,Storage & Organisation,136,112.0,446508.0,26349.0,0.9410,111369.99,6.92
6,Candles & Fragrance,187,144.0,642666.0,58703.0,0.9087,99655.42,6.19
7,Toys & Games,96,85.0,361978.0,42450.0,0.8827,76936.93,4.78
8,Garden & Outdoor,80,66.0,137411.0,19278.0,0.8597,69975.97,4.35
9,Party & Celebration,176,125.0,1017586.0,45541.0,0.9552,64182.76,3.99


## Generate all figures

In [9]:
from src.eda import run as eda_run
for f in eda_run():
    print(f.split("/")[-1])

fig01_demand_seasonality.png
fig02_intermittency.png
fig03_abc_pareto.png
fig04_service_drivers.png
fig05_risk_distribution.png
fig06_excess_by_category.png
fig07_model_comparison.png
fig08_forecast_scenarios.png


→ Continue to `03_demand_forecasting.ipynb`